In [1]:
!pip install openpyxl



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

file_path = "C:/Users/Lenovo/Downloads/data/raw.xlsx"

df = pd.read_excel(file_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Rows: 525461
Columns: 8

Column Names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 5 Rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom



Data Types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

Missing Values:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64


In [3]:
# Step 2: Data Cleaning

# Make a copy of the raw data
cleaned_df = df.copy()

# 1. Remove duplicate rows
print("Duplicate rows before cleaning:", cleaned_df.duplicated().sum())

cleaned_df = cleaned_df.drop_duplicates()

print("Duplicate rows after cleaning:", cleaned_df.duplicated().sum())


# 2. Check missing values
print("\nMissing values before cleaning:")
print(cleaned_df.isnull().sum())


# 3. Remove rows where Customer ID is missing
cleaned_df = cleaned_df.dropna(subset=["Customer ID"])

print("\nRows after removing missing Customer IDs:", len(cleaned_df))


# 4. Remove rows where Description is missing
cleaned_df = cleaned_df.dropna(subset=["Description"])


# 5. Remove transactions with Quantity <= 0
cleaned_df = cleaned_df[cleaned_df["Quantity"] > 0]


# 6. Remove transactions with Price <= 0
cleaned_df = cleaned_df[cleaned_df["Price"] > 0]


# 7. Convert Customer ID to integer
cleaned_df["Customer ID"] = cleaned_df["Customer ID"].astype(int)


# 8. Create Revenue column
cleaned_df["Revenue"] = cleaned_df["Quantity"] * cleaned_df["Price"]


# 9. Check the cleaned dataset
print("\nFinal dataset shape:")
print(cleaned_df.shape)

print("\nMissing values after cleaning:")
print(cleaned_df.isnull().sum())

print("\nFirst 5 rows of cleaned data:")
display(cleaned_df.head())

Duplicate rows before cleaning: 6865
Duplicate rows after cleaning: 0

Missing values before cleaning:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107833
Country             0
dtype: int64

Rows after removing missing Customer IDs: 410763

Final dataset shape:
(400916, 9)

Missing values after cleaning:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
dtype: int64

First 5 rows of cleaned data:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [4]:
# Save cleaned dataset

cleaned_df.to_csv(
    "C:/Users/Lenovo/Downloads/data/cleaned_online_retail.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [5]:
# Step 3: Data Quality & Business Overview

print("===== DATASET OVERVIEW =====")

print("Total transactions:", len(cleaned_df))
print("Unique invoices:", cleaned_df["Invoice"].nunique())
print("Unique customers:", cleaned_df["Customer ID"].nunique())
print("Unique products:", cleaned_df["StockCode"].nunique())
print("Unique countries:", cleaned_df["Country"].nunique())

print("\n===== DATE RANGE =====")

print("Start date:", cleaned_df["InvoiceDate"].min())
print("End date:", cleaned_df["InvoiceDate"].max())

print("\n===== BUSINESS METRICS =====")

print("Total quantity sold:", cleaned_df["Quantity"].sum())

print("Total revenue: £", round(cleaned_df["Revenue"].sum(), 2))

print("Average order value: £", round(
    cleaned_df.groupby("Invoice")["Revenue"].sum().mean(), 2
))

print("\n===== TOP 10 COUNTRIES BY REVENUE =====")

country_revenue = (
    cleaned_df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(country_revenue)

print("\n===== TOP 10 PRODUCTS BY REVENUE =====")

product_revenue = (
    cleaned_df.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(product_revenue)

===== DATASET OVERVIEW =====
Total transactions: 400916
Unique invoices: 19213
Unique customers: 4312
Unique products: 4017
Unique countries: 37

===== DATE RANGE =====
Start date: 2009-12-01 07:45:00
End date: 2010-12-09 20:01:00

===== BUSINESS METRICS =====
Total quantity sold: 5519578
Total revenue: £ 8798233.74
Average order value: £ 457.93

===== TOP 10 COUNTRIES BY REVENUE =====
Country
United Kingdom    7381644.433
EIRE               356041.860
Netherlands        268784.350
Germany            202025.391
France             146107.070
Sweden              53147.990
Denmark             50906.850
Spain               47568.650
Switzerland         43921.390
Australia           31446.800
Name: Revenue, dtype: float64

===== TOP 10 PRODUCTS BY REVENUE =====
Description
WHITE HANGING HEART T-LIGHT HOLDER     151339.16
REGENCY CAKESTAND 3 TIER               143727.60
Manual                                  98531.99
ASSORTED COLOUR BIRD ORNAMENT           70291.03
JUMBO BAG RED RETROSPOT  

In [6]:
# Step 4: Monthly Revenue and Customer Analysis

# Create a Month column
cleaned_df["Month"] = cleaned_df["InvoiceDate"].dt.to_period("M").astype(str)

# Monthly revenue
monthly_revenue = (
    cleaned_df.groupby("Month")["Revenue"]
    .sum()
    .reset_index()
)

# Monthly orders
monthly_orders = (
    cleaned_df.groupby("Month")["Invoice"]
    .nunique()
    .reset_index(name="Orders")
)

# Monthly customers
monthly_customers = (
    cleaned_df.groupby("Month")["Customer ID"]
    .nunique()
    .reset_index(name="Customers")
)

# Combine everything
monthly_analysis = (
    monthly_revenue
    .merge(monthly_orders, on="Month")
    .merge(monthly_customers, on="Month")
)

# Calculate average order value
monthly_analysis["Average Order Value"] = (
    monthly_analysis["Revenue"] / monthly_analysis["Orders"]
)

# Display results
print("===== MONTHLY BUSINESS PERFORMANCE =====")

display(monthly_analysis)

# Highest revenue month
highest_revenue_month = monthly_analysis.loc[
    monthly_analysis["Revenue"].idxmax()
]

print("\n===== HIGHEST REVENUE MONTH =====")
print(highest_revenue_month)

# Lowest revenue month
lowest_revenue_month = monthly_analysis.loc[
    monthly_analysis["Revenue"].idxmin()
]

print("\n===== LOWEST REVENUE MONTH =====")
print(lowest_revenue_month)

===== MONTHLY BUSINESS PERFORMANCE =====


,Month,Revenue,Orders,Customers,Average Order Value
0,2009-12,683504.010,1512,955,452.052917
1,2010-01,555802.672,1011,720,549.755363
2,2010-02,504558.956,1104,772,457.028040
3,2010-03,696978.471,1524,1057,457.334955
4,2010-04,591982.002,1329,942,445.434163
5,2010-05,597833.380,1377,966,434.156412
6,2010-06,636371.130,1497,1041,425.097615
7,2010-07,589736.170,1381,928,427.035605
8,2010-08,602224.600,1293,911,465.757618
9,2010-09,829013.951,1689,1145,490.831232



===== HIGHEST REVENUE MONTH =====
Month                      2010-11
Revenue                1166460.022
Orders                        2587
Customers                     1607
Average Order Value     450.892935
Name: 11, dtype: object

===== LOWEST REVENUE MONTH =====
Month                     2010-12
Revenue                 310656.37
Orders                        776
Customers                     573
Average Order Value    400.330374
Name: 12, dtype: object


In [7]:
# Step 5: Customer-Level Analysis

customer_analysis = (
    cleaned_df.groupby("Customer ID")
    .agg(
        Total_Orders=("Invoice", "nunique"),
        Total_Quantity=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum"),
        First_Purchase=("InvoiceDate", "min"),
        Last_Purchase=("InvoiceDate", "max"),
        Active_Months=("Month", "nunique")
    )
    .reset_index()
)

# Calculate Average Order Value for each customer
customer_analysis["Average_Order_Value"] = (
    customer_analysis["Total_Revenue"] /
    customer_analysis["Total_Orders"]
)

# Sort customers by revenue
customer_analysis = customer_analysis.sort_values(
    "Total_Revenue",
    ascending=False
)

print("===== CUSTOMER ANALYSIS =====")

print("Total customers:", len(customer_analysis))

print("\nTop 10 customers by revenue:")

display(
    customer_analysis.head(10)
)

===== CUSTOMER ANALYSIS =====
Total customers: 4312

Top 10 customers by revenue:


,Customer ID,Total_Orders,Total_Quantity,Total_Revenue,First_Purchase,Last_Purchase,Active_Months,Average_Order_Value
4183,18102,89,124216,349164.35,2009-12-01 09:24:00,2010-12-09 13:44:00,12,3923.194944
1637,14646,78,170278,248396.50,2009-12-02 16:52:00,2010-11-30 16:28:00,12,3184.570513
1269,14156,102,108105,196549.74,2009-12-01 12:30:00,2010-12-03 11:48:00,13,1926.958235
1840,14911,205,69709,152121.22,2009-12-01 11:41:00,2010-12-09 12:17:00,13,742.054732
939,13694,94,125893,131443.19,2009-12-04 15:26:00,2010-12-01 12:12:00,13,1398.331809
3744,17511,31,55107,84541.17,2009-12-02 10:52:00,2010-12-07 09:51:00,13,2727.134516
1951,15061,86,51791,83284.38,2009-12-01 12:18:00,2010-12-07 13:50:00,12,968.423023
3128,16684,27,54555,80489.21,2009-12-07 12:56:00,2010-11-25 11:04:00,9,2981.081852
3177,16754,29,63551,65500.07,2010-03-08 11:32:00,2010-12-02 17:38:00,9,2258.623103
4065,17949,74,30112,60117.60,2009-12-02 11:07:00,2010-12-03 13:12:00,13,812.400000


In [8]:
# Customer Revenue Statistics

print("===== CUSTOMER REVENUE STATISTICS =====")

print(
    customer_analysis["Total_Revenue"].describe()
)

print("\n===== TOP 10 CUSTOMERS BY REVENUE =====")

top_customers = customer_analysis[
    ["Customer ID", "Total_Orders", "Total_Quantity", "Total_Revenue", "Average_Order_Value"]
].head(10)

display(top_customers)

===== CUSTOMER REVENUE STATISTICS =====
count      4312.000000
mean       2040.406712
std        8911.755977
min           2.950000
25%         307.187500
50%         701.615000
75%        1714.932500
max      349164.350000
Name: Total_Revenue, dtype: float64

===== TOP 10 CUSTOMERS BY REVENUE =====


,Customer ID,Total_Orders,Total_Quantity,Total_Revenue,Average_Order_Value
4183,18102,89,124216,349164.35,3923.194944
1637,14646,78,170278,248396.50,3184.570513
1269,14156,102,108105,196549.74,1926.958235
1840,14911,205,69709,152121.22,742.054732
939,13694,94,125893,131443.19,1398.331809
3744,17511,31,55107,84541.17,2727.134516
1951,15061,86,51791,83284.38,968.423023
3128,16684,27,54555,80489.21,2981.081852
3177,16754,29,63551,65500.07,2258.623103
4065,17949,74,30112,60117.60,812.400000


In [9]:
# Step 6: Customer Revenue Concentration

# Sort customers from highest to lowest revenue
customer_revenue = customer_analysis.sort_values(
    "Total_Revenue",
    ascending=False
).reset_index(drop=True)

total_revenue = customer_revenue["Total_Revenue"].sum()
total_customers = len(customer_revenue)

print("===== CUSTOMER REVENUE CONCENTRATION =====")

for percentage in [1, 5, 10, 20]:

    number_of_customers = max(
        1,
        int(total_customers * percentage / 100)
    )

    top_revenue = customer_revenue.head(
        number_of_customers
    )["Total_Revenue"].sum()

    revenue_share = (
        top_revenue / total_revenue
    ) * 100

    print(
        f"Top {percentage}% of customers "
        f"({number_of_customers} customers): "
        f"£{top_revenue:,.2f} "
        f"= {revenue_share:.2f}% of total revenue"
    )

===== CUSTOMER REVENUE CONCENTRATION =====
Top 1% of customers (43 customers): £2,514,680.76 = 28.58% of total revenue
Top 5% of customers (215 customers): £4,253,090.29 = 48.34% of total revenue
Top 10% of customers (431 customers): £5,271,661.02 = 59.92% of total revenue
Top 20% of customers (862 customers): £6,469,005.31 = 73.53% of total revenue


In [10]:
# Top customer contribution

top_customer_revenue = customer_revenue.iloc[0]["Total_Revenue"]

top_customer_share = (
    top_customer_revenue / total_revenue
) * 100

print(
    f"Top customer revenue: £{top_customer_revenue:,.2f}"
)

print(
    f"Top customer revenue share: {top_customer_share:.2f}%"
)

Top customer revenue: £349,164.35
Top customer revenue share: 3.97%


In [11]:
# Step 7: RFM Analysis

# Set the analysis date as one day after the last transaction
analysis_date = cleaned_df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    cleaned_df.groupby("Customer ID")
    .agg(
        Recency=("InvoiceDate", lambda x:
                 (analysis_date - x.max()).days),

        Frequency=("Invoice", "nunique"),

        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

print("===== RFM DATA =====")

print("Total customers:", len(rfm))

display(rfm.head(10))

===== RFM DATA =====
Total customers: 4312


,Customer ID,Recency,Frequency,Monetary
0,12346,165,11,372.86
1,12347,3,2,1323.32
2,12348,74,1,222.16
3,12349,43,3,2671.14
4,12351,11,1,300.93
5,12352,11,2,343.80
6,12353,44,1,317.76
7,12355,203,1,488.21
8,12356,16,3,3560.30
9,12357,24,2,12079.99


In [12]:
# Create RFM scores from 1 to 5

rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Combined RFM score
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

print("===== RFM SCORES =====")

display(rfm.head(10))

===== RFM SCORES =====


,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346,165,11,372.86,2,5,2,252
1,12347,3,2,1323.32,5,2,4,524
2,12348,74,1,222.16,2,1,1,211
3,12349,43,3,2671.14,3,3,5,335
4,12351,11,1,300.93,5,1,2,512
5,12352,11,2,343.80,5,2,2,522
6,12353,44,1,317.76,3,1,2,312
7,12355,203,1,488.21,1,1,2,112
8,12356,16,3,3560.30,4,3,5,435
9,12357,24,2,12079.99,4,2,5,425


In [13]:
# Step 8: Create Customer Segments

def segment_customer(row):

    R = row["R_Score"]
    F = row["F_Score"]
    M = row["M_Score"]

    # Champions
    if R >= 4 and F >= 4 and M >= 4:
        return "Champions"

    # Loyal Customers
    elif R >= 3 and F >= 4 and M >= 3:
        return "Loyal Customers"

    # Big Spenders
    elif R >= 3 and M >= 4:
        return "Big Spenders"

    # Potential Loyalists
    elif R >= 4 and F >= 2:
        return "Potential Loyalists"

    # New Customers
    elif R >= 4 and F <= 2:
        return "New Customers"

    # At Risk
    elif R <= 2 and F >= 3 and M >= 3:
        return "At Risk"

    # Can't Lose Them
    elif R <= 2 and F >= 4:
        return "Can't Lose Them"

    # Lost Customers
    elif R <= 2 and F <= 2 and M <= 2:
        return "Lost Customers"

    # Others
    else:
        return "Regular Customers"


rfm["Customer_Segment"] = rfm.apply(
    segment_customer,
    axis=1
)

print("===== CUSTOMER SEGMENTS =====")

segment_summary = (
    rfm.groupby("Customer_Segment")
    .agg(
        Customers=("Customer ID", "count"),
        Revenue=("Monetary", "sum"),
        Average_Revenue=("Monetary", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

display(segment_summary)

===== CUSTOMER SEGMENTS =====


,Customers,Revenue,Average_Revenue
Customer_Segment,,,
Champions,926,5712430.697,6168.931638
At Risk,509,981368.241,1928.031908
Loyal Customers,432,827091.522,1914.563708
Regular Customers,792,454634.863,574.033918
Big Spenders,209,383621.740,1835.510718
Potential Loyalists,449,202427.370,450.840468
Lost Customers,830,186255.161,224.403808
New Customers,129,37503.090,290.721628
Can't Lose Them,36,12901.060,358.362778


In [14]:
print("===== CUSTOMER SEGMENT DISTRIBUTION =====")

segment_count = (
    rfm["Customer_Segment"]
    .value_counts()
    .reset_index()
)

segment_count.columns = ["Customer Segment", "Customers"]

display(segment_count)

===== CUSTOMER SEGMENT DISTRIBUTION =====


,Customer Segment,Customers
0,Champions,926
1,Lost Customers,830
2,Regular Customers,792
3,At Risk,509
4,Potential Loyalists,449
5,Loyal Customers,432
6,Big Spenders,209
7,New Customers,129
8,Can't Lose Them,36


In [15]:
# Step 9: Revenue by Customer Segment

segment_revenue = (
    rfm.groupby("Customer_Segment")
    .agg(
        Customers=("Customer ID", "count"),
        Total_Revenue=("Monetary", "sum"),
        Average_Revenue=("Monetary", "mean")
    )
    .reset_index()
)

segment_revenue["Revenue_Share_%"] = (
    segment_revenue["Total_Revenue"]
    / segment_revenue["Total_Revenue"].sum()
    * 100
)

segment_revenue = segment_revenue.sort_values(
    "Total_Revenue",
    ascending=False
)

print("===== REVENUE BY CUSTOMER SEGMENT =====")

display(segment_revenue)

===== REVENUE BY CUSTOMER SEGMENT =====


,Customer_Segment,Customers,Total_Revenue,Average_Revenue,Revenue_Share_%
3,Champions,926,5712430.697,6168.931638,64.927017
0,At Risk,509,981368.241,1928.031908,11.154151
5,Loyal Customers,432,827091.522,1914.563708,9.400654
8,Regular Customers,792,454634.863,574.033918,5.167342
1,Big Spenders,209,383621.740,1835.510718,4.360213
7,Potential Loyalists,449,202427.370,450.840468,2.300773
4,Lost Customers,830,186255.161,224.403808,2.116961
6,New Customers,129,37503.090,290.721628,0.426257
2,Can't Lose Them,36,12901.060,358.362778,0.146632


In [16]:
# Top revenue-generating segment

top_segment = segment_revenue.iloc[0]

print("===== TOP CUSTOMER SEGMENT =====")
print("Segment:", top_segment["Customer_Segment"])
print("Customers:", int(top_segment["Customers"]))
print("Revenue: £", round(top_segment["Total_Revenue"], 2))
print("Revenue Share:", round(top_segment["Revenue_Share_%"], 2), "%")

===== TOP CUSTOMER SEGMENT =====
Segment: Champions
Customers: 926
Revenue: £ 5712430.7
Revenue Share: 64.93 %


In [17]:
# Step 10: Repeat vs One-Time Customers

customer_orders = (
    cleaned_df.groupby("Customer ID")["Invoice"]
    .nunique()
    .reset_index(name="Orders")
)

customer_orders["Customer_Type"] = customer_orders["Orders"].apply(
    lambda x: "Repeat Customer" if x > 1 else "One-Time Customer"
)

print("===== CUSTOMER TYPE DISTRIBUTION =====")

customer_type_count = (
    customer_orders["Customer_Type"]
    .value_counts()
    .reset_index()
)

customer_type_count.columns = ["Customer Type", "Customers"]

display(customer_type_count)

===== CUSTOMER TYPE DISTRIBUTION =====


,Customer Type,Customers
0,Repeat Customer,2893
1,One-Time Customer,1419


In [18]:
# Revenue generated by customer type

customer_revenue_type = (
    cleaned_df.groupby("Customer ID")["Revenue"]
    .sum()
    .reset_index(name="Revenue")
)

customer_revenue_type = customer_revenue_type.merge(
    customer_orders,
    on="Customer ID"
)

revenue_by_type = (
    customer_revenue_type
    .groupby("Customer_Type")
    .agg(
        Customers=("Customer ID", "count"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

revenue_by_type["Revenue_Share_%"] = (
    revenue_by_type["Revenue"]
    / revenue_by_type["Revenue"].sum()
    * 100
)

print("===== REVENUE BY CUSTOMER TYPE =====")

display(revenue_by_type)

===== REVENUE BY CUSTOMER TYPE =====


,Customer_Type,Customers,Revenue,Revenue_Share_%
0,One-Time Customer,1419,496304.474,5.640956
1,Repeat Customer,2893,8301929.270,94.359044


In [19]:
# Repeat customer rate

repeat_customers = (
    customer_orders["Customer_Type"] == "Repeat Customer"
).sum()

total_customers = len(customer_orders)

repeat_rate = (
    repeat_customers / total_customers
) * 100

print("===== REPEAT CUSTOMER RATE =====")
print("Repeat customers:", repeat_customers)
print("Total customers:", total_customers)
print("Repeat customer rate:", round(repeat_rate, 2), "%")

===== REPEAT CUSTOMER RATE =====
Repeat customers: 2893
Total customers: 4312
Repeat customer rate: 67.09 %


In [20]:
# Step 11: Monthly Customer & Revenue Trends

monthly_customer_analysis = (
    cleaned_df.groupby("Month")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Invoice", "nunique"),
        Customers=("Customer ID", "nunique")
    )
    .reset_index()
)

# Calculate repeat customers for each month
repeat_monthly = (
    cleaned_df.groupby(["Month", "Customer ID"])["Invoice"]
    .nunique()
    .reset_index(name="Customer_Orders")
)

repeat_monthly["Repeat_Customer"] = (
    repeat_monthly["Customer_Orders"] > 1
)

monthly_repeat = (
    repeat_monthly.groupby("Month")["Repeat_Customer"]
    .sum()
    .reset_index(name="Repeat_Customers")
)

monthly_customer_analysis = monthly_customer_analysis.merge(
    monthly_repeat,
    on="Month"
)

monthly_customer_analysis["Repeat_Customer_Rate_%"] = (
    monthly_customer_analysis["Repeat_Customers"]
    / monthly_customer_analysis["Customers"]
    * 100
)

monthly_customer_analysis["Revenue_per_Customer"] = (
    monthly_customer_analysis["Revenue"]
    / monthly_customer_analysis["Customers"]
)

print("===== MONTHLY CUSTOMER ANALYSIS =====")

display(monthly_customer_analysis)

===== MONTHLY CUSTOMER ANALYSIS =====


,Month,Revenue,Orders,Customers,Repeat_Customers,Repeat_Customer_Rate_%,Revenue_per_Customer
0,2009-12,683504.010,1512,955,286,29.947644,715.711005
1,2010-01,555802.672,1011,720,148,20.555556,771.948156
2,2010-02,504558.956,1104,772,175,22.668394,653.573777
3,2010-03,696978.471,1524,1057,237,22.421949,659.393066
4,2010-04,591982.002,1329,942,202,21.443737,628.431000
5,2010-05,597833.380,1377,966,228,23.602484,618.875135
6,2010-06,636371.130,1497,1041,257,24.687800,611.307522
7,2010-07,589736.170,1381,928,255,27.478448,635.491563
8,2010-08,602224.600,1293,911,219,24.039517,661.058836
9,2010-09,829013.951,1689,1145,270,23.580786,724.029652


In [21]:
# Step 12: Cohort Retention Analysis

cohort_df = cleaned_df.copy()

# Purchase month
cohort_df["Purchase_Month"] = (
    cohort_df["InvoiceDate"]
    .dt.to_period("M")
)

# First purchase month for each customer
first_purchase = (
    cohort_df.groupby("Customer ID")["Purchase_Month"]
    .min()
    .reset_index(name="Cohort_Month")
)

# Add cohort month to transaction data
cohort_df = cohort_df.merge(
    first_purchase,
    on="Customer ID",
    how="left"
)

# Calculate months since first purchase
cohort_df["Cohort_Index"] = (
    (cohort_df["Purchase_Month"].dt.year -
     cohort_df["Cohort_Month"].dt.year) * 12
    +
    (cohort_df["Purchase_Month"].dt.month -
     cohort_df["Cohort_Month"].dt.month)
)

# Count unique customers in each cohort/month
cohort_data = (
    cohort_df
    .groupby(["Cohort_Month", "Cohort_Index"])["Customer ID"]
    .nunique()
    .reset_index()
)

# Convert into retention matrix
cohort_pivot = cohort_data.pivot(
    index="Cohort_Month",
    columns="Cohort_Index",
    values="Customer ID"
)

# Calculate cohort size
cohort_size = (
    cohort_pivot.iloc[:, 0]
)

# Calculate retention percentage
cohort_retention = (
    cohort_pivot
    .divide(cohort_size, axis=0)
    * 100
)

print("===== COHORT RETENTION =====")

display(cohort_retention.round(2))


===== COHORT RETENTION =====


Cohort_Index,0,1,2,3,4,5,6,7,8,9,10,11,12
Cohort_Month,,,,,,,,,,,,,
2009-12,100.0,35.29,33.40,42.51,38.01,35.92,37.70,34.24,33.61,36.23,42.20,49.53,24.82
2010-01,100.0,20.63,31.07,30.55,26.37,30.03,25.85,22.98,27.94,31.85,30.29,9.92,NaN
2010-02,100.0,23.80,22.46,29.14,24.60,20.05,19.25,28.61,25.40,27.54,7.22,NaN,NaN
2010-03,100.0,18.96,23.02,24.15,23.25,20.32,24.60,30.25,27.54,7.90,NaN,NaN,NaN
2010-04,100.0,19.39,19.39,16.33,18.37,22.45,27.55,26.19,6.80,NaN,NaN,NaN,NaN
2010-05,100.0,15.75,16.93,17.32,17.72,25.59,21.26,7.87,NaN,NaN,NaN,NaN,NaN
2010-06,100.0,17.41,18.89,20.37,22.96,28.52,6.67,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,100.0,15.59,18.28,29.57,29.03,10.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,100.0,20.37,29.63,32.10,11.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Step 13: Cohort Retention Summary

cohort_summary = cohort_retention[
    [1, 3, 6]
].copy()

cohort_summary.columns = [
    "Month_1_Retention_%",
    "Month_3_Retention_%",
    "Month_6_Retention_%"
]

print("===== COHORT RETENTION SUMMARY =====")

display(cohort_summary.round(2))

===== COHORT RETENTION SUMMARY =====


,Month_1_Retention_%,Month_3_Retention_%,Month_6_Retention_%
Cohort_Month,,,
2009-12,35.29,42.51,37.70
2010-01,20.63,30.55,25.85
2010-02,23.80,29.14,19.25
2010-03,18.96,24.15,24.60
2010-04,19.39,16.33,27.55
2010-05,15.75,17.32,21.26
2010-06,17.41,20.37,6.67
2010-07,15.59,29.57,NaN
2010-08,20.37,32.10,NaN


In [23]:
# Best cohorts by Month 1 retention

print("===== BEST MONTH 1 RETENTION =====")

best_month1 = (
    cohort_summary["Month_1_Retention_%"]
    .dropna()
    .sort_values(ascending=False)
    .head(5)
)

display(best_month1.round(2))

===== BEST MONTH 1 RETENTION =====


Cohort_Month
2009-12    35.29
2010-10    25.73
2010-02    23.80
2010-09    22.63
2010-01    20.63
Freq: M, Name: Month_1_Retention_%, dtype: float64

In [24]:
# Best cohorts by Month 3 retention

print("===== BEST MONTH 3 RETENTION =====")

best_month3 = (
    cohort_summary["Month_3_Retention_%"]
    .dropna()
    .sort_values(ascending=False)
    .head(5)
)

display(best_month3.round(2))

===== BEST MONTH 3 RETENTION =====


Cohort_Month
2009-12    42.51
2010-08    32.10
2010-01    30.55
2010-07    29.57
2010-02    29.14
Freq: M, Name: Month_3_Retention_%, dtype: float64

In [25]:
# Best cohorts by Month 6 retention

print("===== BEST MONTH 6 RETENTION =====")

best_month6 = (
    cohort_summary["Month_6_Retention_%"]
    .dropna()
    .sort_values(ascending=False)
    .head(5)
)

display(best_month6.round(2))

===== BEST MONTH 6 RETENTION =====


Cohort_Month
2009-12    37.70
2010-04    27.55
2010-01    25.85
2010-03    24.60
2010-05    21.26
Freq: M, Name: Month_6_Retention_%, dtype: float64

In [26]:
# Step 14: Product Performance Analysis

product_analysis = (
    cleaned_df.groupby("Description")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity_Sold=("Quantity", "sum"),
        Orders=("Invoice", "nunique"),
        Customers=("Customer ID", "nunique"),
        Average_Price=("Price", "mean")
    )
    .reset_index()
)

# Sort by revenue
top_products_revenue = product_analysis.sort_values(
    "Revenue",
    ascending=False
).head(20)

print("===== TOP 20 PRODUCTS BY REVENUE =====")

display(top_products_revenue)

===== TOP 20 PRODUCTS BY REVENUE =====


,Description,Revenue,Quantity_Sold,Orders,Customers,Average_Price
4260,WHITE HANGING HEART T-LIGHT HOLDER,151339.16,56814,3021,1141,2.854422
3252,REGENCY CAKESTAND 3 TIER,143727.60,12484,1687,818,12.436667
2448,Manual,98531.99,2568,382,283,229.954024
259,ASSORTED COLOUR BIRD ORNAMENT,70291.03,44431,1333,627,1.671697
2069,JUMBO BAG RED RETROSPOT,51644.25,29519,1060,539,1.907302
3049,POSTAGE,48741.08,2212,738,230,27.052615
3369,ROTATING SILVER ANGELS T-LIGHT HLDR,40156.05,21579,244,177,2.505333
2666,PAPER CHAIN KIT 50'S CHRISTMAS,36871.55,13839,796,505,2.896581
2680,PARTY BUNTING,35017.30,8312,710,391,4.569348
1297,EDWARDIAN PARASOL NATURAL,34044.75,7201,496,245,5.729175


In [27]:
print("===== TOP 20 PRODUCTS BY QUANTITY SOLD =====")

top_products_quantity = (
    product_analysis
    .sort_values("Quantity_Sold", ascending=False)
    .head(20)
)

display(top_products_quantity)

===== TOP 20 PRODUCTS BY QUANTITY SOLD =====


,Description,Revenue,Quantity_Sold,Orders,Customers,Average_Price
4260,WHITE HANGING HEART T-LIGHT HOLDER,151339.16,56814,3021,1141,2.854422
4366,WORLD WAR 2 GLIDERS ASSTD DESIGNS,11285.54,54610,461,282,0.231104
690,BROCADE RING PURSE,8847.66,48166,188,127,0.203646
2632,PACK OF 72 RETRO SPOT CAKE CASES,21838.23,45129,1160,632,0.536441
259,ASSORTED COLOUR BIRD ORNAMENT,70291.03,44431,1333,627,1.671697
138,60 TEATIME FAIRY CAKE CASES,17436.30,35656,1136,600,0.540539
2630,PACK OF 60 PINK PAISLEY CAKE CASES,15045.10,30871,928,518,0.540222
2069,JUMBO BAG RED RETROSPOT,51644.25,29519,1060,539,1.907302
3852,SMALL POPCORN HOLDER,19518.21,25703,678,314,0.837299
413,BLACK AND WHITE PAISLEY FLOWER MUG,3019.57,25685,18,13,1.045000


In [28]:
print("===== TOP 20 PRODUCTS BY NUMBER OF CUSTOMERS =====")

top_products_customers = (
    product_analysis
    .sort_values("Customers", ascending=False)
    .head(20)
)

display(top_products_customers)

===== TOP 20 PRODUCTS BY NUMBER OF CUSTOMERS =====


,Description,Revenue,Quantity_Sold,Orders,Customers,Average_Price
4260,WHITE HANGING HEART T-LIGHT HOLDER,151339.16,56814,3021,1141,2.854422
3252,REGENCY CAKESTAND 3 TIER,143727.60,12484,1687,818,12.436667
331,BAKING SET 9 PIECE RETROSPOT,20117.81,4249,888,632,4.915944
2632,PACK OF 72 RETRO SPOT CAKE CASES,21838.23,45129,1160,632,0.536441
259,ASSORTED COLOUR BIRD ORNAMENT,70291.03,44431,1333,627,1.671697
138,60 TEATIME FAIRY CAKE CASES,17436.30,35656,1136,600,0.540539
1933,HEART OF WICKER LARGE,23551.18,8246,997,570,2.934255
3933,STRAWBERRY CERAMIC TRINKET BOX,29694.86,25330,1335,555,1.232551
3147,RED HANGING HEART T-LIGHT HOLDER,27300.61,9855,994,549,2.893425
2069,JUMBO BAG RED RETROSPOT,51644.25,29519,1060,539,1.907302


In [29]:
# Step 15: Country Performance Analysis

country_analysis = (
    cleaned_df.groupby("Country")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Invoice", "nunique"),
        Customers=("Customer ID", "nunique"),
        Quantity_Sold=("Quantity", "sum")
    )
    .reset_index()
)

country_analysis["Revenue_per_Customer"] = (
    country_analysis["Revenue"] /
    country_analysis["Customers"]
)

country_analysis = country_analysis.sort_values(
    "Revenue",
    ascending=False
)

print("===== COUNTRY PERFORMANCE =====")

display(country_analysis)

===== COUNTRY PERFORMANCE =====


,Country,Revenue,Orders,Customers,Quantity_Sold,Revenue_per_Customer
34,United Kingdom,7381644.433,17612,3969,4430148,1859.824750
9,EIRE,356041.860,316,5,181413,71208.372000
21,Netherlands,268784.350,135,22,183615,12217.470455
12,Germany,202025.391,347,67,108615,3015.304343
11,France,146107.070,236,47,162048,3108.661064
29,Sweden,53147.990,68,16,52417,3321.749375
8,Denmark,50906.850,26,9,229690,5656.316667
28,Spain,47568.650,66,25,22841,1902.746000
30,Switzerland,43921.390,40,14,22255,3137.242143
0,Australia,31446.800,40,15,20189,2096.453333


In [30]:
print("===== TOP 10 COUNTRIES BY REVENUE =====")

display(
    country_analysis.head(10)
)

===== TOP 10 COUNTRIES BY REVENUE =====


,Country,Revenue,Orders,Customers,Quantity_Sold,Revenue_per_Customer
34,United Kingdom,7381644.433,17612,3969,4430148,1859.824750
9,EIRE,356041.860,316,5,181413,71208.372000
21,Netherlands,268784.350,135,22,183615,12217.470455
12,Germany,202025.391,347,67,108615,3015.304343
11,France,146107.070,236,47,162048,3108.661064
29,Sweden,53147.990,68,16,52417,3321.749375
8,Denmark,50906.850,26,9,229690,5656.316667
28,Spain,47568.650,66,25,22841,1902.746000
30,Switzerland,43921.390,40,14,22255,3137.242143
0,Australia,31446.800,40,15,20189,2096.453333


In [31]:
print("===== TOP 10 COUNTRIES BY CUSTOMER COUNT =====")

display(
    country_analysis
    .sort_values("Customers", ascending=False)
    .head(10)
)

===== TOP 10 COUNTRIES BY CUSTOMER COUNT =====


,Country,Revenue,Orders,Customers,Quantity_Sold,Revenue_per_Customer
34,United Kingdom,7381644.433,17612,3969,4430148,1859.824750
12,Germany,202025.391,347,67,108615,3015.304343
11,France,146107.070,236,47,162048,3108.661064
28,Spain,47568.650,66,25,22841,1902.746000
21,Netherlands,268784.350,135,22,183615,12217.470455
25,Portugal,23843.460,40,18,11988,1324.636667
3,Belgium,24537.580,52,17,12059,1443.387059
29,Sweden,53147.990,68,16,52417,3321.749375
0,Australia,31446.800,40,15,20189,2096.453333
30,Switzerland,43921.390,40,14,22255,3137.242143
